# Next-number prediction with the hand-built LSTM cell

Reuse the scratch `LSTM` cell from `lstm_v2.ipynb` to predict the **next number** in a
sequence, e.g. `[10, 11, 12, 13] -> 14`.

The cell's gate weights are fixed (we never derived backprop for it), so we treat the
LSTM as a frozen **feature extractor**: feed a sequence through it one number at a time,
collect the hidden states it produces, and train only a small **linear readout** on top
to map those features to the next number. (This is the idea behind *reservoir computing* /
echo-state networks.)

In [9]:
import numpy as np

## 1. The LSTM cell (unchanged from `lstm_v2`)

In [10]:
class LSTM:
    """A single LSTM cell implemented from scratch with numpy.

    Each weight vector is [w_h, w_x] and is applied to the concatenated
    input [h_{t-1}, x_t].
    """

    def __init__(self, W_f, W_i, W_c, W_o, b_f, b_i, b_c, b_o):
        self.W_f, self.W_i, self.W_c, self.W_o = W_f, W_i, W_c, W_o
        self.b_f, self.b_i, self.b_c, self.b_o = b_f, b_i, b_c, b_o

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    @staticmethod
    def tanh(x):
        return np.tanh(x)

    def forward(self, X_t, H_t_minus_1, C_t_minus_1):
        """Run one time step. Returns (H_t, C_t)."""
        inputs = np.array([H_t_minus_1, X_t])

        F_t = self.sigmoid(np.dot(self.W_f, inputs) + self.b_f)
        I_t = self.sigmoid(np.dot(self.W_i, inputs) + self.b_i)
        C_t_bar = self.tanh(np.dot(self.W_c, inputs) + self.b_c)
        C_t = F_t * C_t_minus_1 + I_t * C_t_bar
        O_t = self.sigmoid(np.dot(self.W_o, inputs) + self.b_o)
        H_t = O_t * self.tanh(C_t)
        return H_t, C_t


# Same fixed weights/biases used in lstm_v2
lstm = LSTM(
    W_f=np.array([2.70, 1.63]), W_i=np.array([2.0, 1.65]),
    W_c=np.array([1.41, 0.94]), W_o=np.array([4.38, -0.19]),
    b_f=1.62, b_i=0.62, b_c=-0.32, b_o=0.59,
)

## 2. Run a sequence through the cell

Feed the numbers in one at a time, carrying the hidden state `H` and cell state `C`
forward (both start at 0). We keep the hidden state after **every** step plus the final
cell state &mdash; that vector is our feature representation of the whole sequence.

Numbers are divided by a `SCALE` so the inputs stay in a range the sigmoid/tanh gates
respond to.

In [11]:
WINDOW = 4      # how many numbers we look at
SCALE = 50.0    # keep inputs roughly in [0, 1]

def sequence_features(seq):
    """Run `seq` through the frozen LSTM cell; return [H_1..H_n, C_n]."""
    H, C = 0.0, 0.0
    hidden = []
    for x in seq:
        H, C = lstm.forward(x, H, C)
        hidden.append(H)
    return np.array(hidden + [C])

# Quick look at what the cell produces for one sequence
demo = np.array([10, 11, 12, 13]) / SCALE
print("features for [10, 11, 12, 13]:", np.round(sequence_features(demo), 4))

features for [10, 11, 12, 13]: [-0.0599 -0.1229 -0.1745 -0.2083 -0.5084]
